In [1]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [2]:
# local embedding model import

MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10785.50it/s]


In [3]:
documents = [
    Document(page_content="RAG means Retrieval Augmented Generation."),
    Document(page_content="Retrievers find relevant documents."),
    Document(page_content="Vector databases store embeddings."),
    Document(page_content="Cricket is a popular sport."),
]

vector_store = Chroma(
    collection_name="score_threshold_doc",
    embedding_function=embeddings,
    persist_directory="../all_vector_stores/chroma_score_threshold"
)

vector_store.add_documents(documents)

['c6448937-d642-40ac-8a2d-087d220bcfbf',
 '8fb18007-173b-449f-861f-8823e171116f',
 '69976925-cf41-4305-a7c1-b1cf8b0961b8',
 'b1185be0-bb77-4113-ad21-1f37e2946d38']

In [4]:
# score threshold ব্যবহার করছি
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    
    search_kwargs={
        "k": 3,                  # k = maximum document count
        "score_threshold": 0.4   # score_threshold = এর নিচের score-এর document বাদ যাবে
    }
)

query = "What is RAG?"

results = retriever.invoke(query)

/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='c6448937-d642-40ac-8a2d-087d220bcfbf', metadata={}, page_content='RAG means Retrieval Augmented Generation.'), 0.4273085381402749), (Document(id='b1185be0-bb77-4113-ad21-1f37e2946d38', metadata={}, page_content='Cricket is a popular sport.'), -0.2762262976349743), (Document(id='69976925-cf41-4305-a7c1-b1cf8b0961b8', metadata={}, page_content='Vector databases store embeddings.'), -0.3513209392070775)]
  self.vectorstore.similarity_search_with_relevance_scores(


In [5]:
print("Query:", query)
print("-" * 50)

if not results:
    print("No document crossed the similarity threshold.")

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Query: What is RAG?
--------------------------------------------------
1. RAG means Retrieval Augmented Generation.
